In [1]:
from transformers import pipeline
import warnings
warnings.filterwarnings('ignore')

print("Hugging Face Transformers yüklendi!")
print()
print("Bugün yapacaklarımız:")
print("1. Hazır pipeline ile sentiment analizi")
print("2. Tokenizer — metin nasıl sayıya dönüşür")
print("3. BERT ile özellik çıkarımı")
print("4. BERT'i kendi verimize fine-tune etmek")

C:\Users\PC\AppData\Roaming\Python\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Hugging Face Transformers yüklendi!

Bugün yapacaklarımız:
1. Hazır pipeline ile sentiment analizi
2. Tokenizer — metin nasıl sayıya dönüşür
3. BERT ile özellik çıkarımı
4. BERT'i kendi verimize fine-tune etmek


In [2]:
# Hazır sentiment analizi pipeline'ı
sentiment = pipeline("sentiment-analysis")

cümleler = [
    "I absolutely love this product, it's amazing!",
    "This is the worst experience I've ever had.",
    "The weather is okay today, nothing special.",
    "I can't believe how beautiful this is!",
    "Terrible service, never coming back."
]

print("Sentiment Analizi Sonuçları:")
print("-" * 50)
for cümle in cümleler:
    sonuç = sentiment(cümle)[0]
    emoji = "😊" if sonuç['label'] == 'POSITIVE' else "😞"
    print(f"{emoji} {sonuç['label']} ({sonuç['score']:.2%})")
    print(f"   '{cümle}'")
    print()

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.
Loading weights: 100%|██████████| 104/104 [00:00<00:00, 3411.36it/s]


Sentiment Analizi Sonuçları:
--------------------------------------------------
😊 POSITIVE (99.99%)
   'I absolutely love this product, it's amazing!'

😞 NEGATIVE (99.98%)
   'This is the worst experience I've ever had.'

😞 NEGATIVE (96.45%)
   'The weather is okay today, nothing special.'

😊 POSITIVE (99.98%)
   'I can't believe how beautiful this is!'

😞 NEGATIVE (99.20%)
   'Terrible service, never coming back.'



In [3]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

cümle = "I love deep learning!"

# Tokenize et
tokens = tokenizer.tokenize(cümle)
token_ids = tokenizer.encode(cümle)
decoded = tokenizer.decode(token_ids)

print("Orijinal cümle:", cümle)
print("\nTokenlar:", tokens)
print("Token ID'leri:", token_ids)
print("Geri çevrilmiş:", decoded)
print("\nVocabulary boyutu:", tokenizer.vocab_size)

Orijinal cümle: I love deep learning!

Tokenlar: ['i', 'love', 'deep', 'learning', '!']
Token ID'leri: [101, 1045, 2293, 2784, 4083, 999, 102]
Geri çevrilmiş: [CLS] i love deep learning! [SEP]

Vocabulary boyutu: 30522


In [4]:
# Bilinmeyen ve karmaşık kelimeleri test et
örnekler = [
    "ChatGPT is revolutionizing AI!",
    "I love playing football",
    "antidisestablishmentarianism",  # çok uzun kelime
    "😊 emoji test 🚀"
]

print("Tokenizasyon Örnekleri:")
print("-" * 50)
for örnek in örnekler:
    tokens = tokenizer.tokenize(örnek)
    print(f"\nGirdi: '{örnek}'")
    print(f"Token sayısı: {len(tokens)}")
    print(f"Tokenlar: {tokens}")

Tokenizasyon Örnekleri:
--------------------------------------------------

Girdi: 'ChatGPT is revolutionizing AI!'
Token sayısı: 8
Tokenlar: ['chat', '##gp', '##t', 'is', 'revolution', '##izing', 'ai', '!']

Girdi: 'I love playing football'
Token sayısı: 4
Tokenlar: ['i', 'love', 'playing', 'football']

Girdi: 'antidisestablishmentarianism'
Token sayısı: 8
Tokenlar: ['anti', '##dis', '##est', '##ab', '##lish', '##ment', '##arian', '##ism']

Girdi: '😊 emoji test 🚀'
Token sayısı: 6
Tokenlar: ['[UNK]', 'em', '##oj', '##i', 'test', '[UNK]']


In [5]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch

# Sıfırdan fine-tuning yerine hazır modeli kullanalım
# IMDb film yorumları üzerinde eğitilmiş model
classifier = pipeline(
    "text-classification",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

# Kendi cümlelerimizi test edelim
test_cümleleri = [
    "This movie was absolutely fantastic, best film of the year!",
    "I wasted 2 hours of my life watching this garbage.",
    "The acting was decent but the plot was confusing.",
    "Outstanding performance by all actors, highly recommended!",
    "Boring, predictable, and a complete waste of money."
]

print("Film Yorum Analizi:")
print("=" * 60)
for cümle in test_cümleleri:
    sonuç = classifier(cümle)[0]
    bar = "█" * int(sonuç['score'] * 20)
    emoji = "⭐" if sonuç['label'] == 'POSITIVE' else "💔"
    print(f"\n{emoji} {sonuç['label']} {bar} {sonuç['score']:.2%}")
    print(f"   {cümle}")

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 1612.44it/s]


Film Yorum Analizi:

⭐ POSITIVE ███████████████████ 99.99%
   This movie was absolutely fantastic, best film of the year!

💔 NEGATIVE ███████████████████ 99.97%
   I wasted 2 hours of my life watching this garbage.

💔 NEGATIVE ███████████████████ 99.57%
   The acting was decent but the plot was confusing.

⭐ POSITIVE ███████████████████ 99.99%
   Outstanding performance by all actors, highly recommended!

💔 NEGATIVE ███████████████████ 99.98%
   Boring, predictable, and a complete waste of money.


In [7]:
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModelForSequenceClassification
from torch.optim import AdamW

# Küçük bir Türkçe/İngilizce sentiment veri seti
veri = [
    ("This product is amazing, I love it!", 1),
    ("Absolutely terrible, do not buy!", 0),
    ("Great quality and fast delivery!", 1),
    ("Broken on arrival, very disappointed.", 0),
    ("Best purchase I've made this year!", 1),
    ("Complete waste of money, avoid!", 0),
    ("Exceeded my expectations, highly recommend!", 1),
    ("Poor quality, fell apart after one day.", 0),
    ("Fantastic product, works perfectly!", 1),
    ("Awful experience, never again.", 0),
    ("Super happy with this purchase!", 1),
    ("Defective product, terrible support.", 0),
    ("Outstanding quality, worth every penny!", 1),
    ("Garbage product, total scam.", 0),
    ("Love it! Exactly as described.", 1),
    ("Very poor quality, not recommended.", 0),
]

print(f"Veri seti: {len(veri)} örnek")
print(f"Pozitif: {sum(1 for _, l in veri if l == 1)}")
print(f"Negatif: {sum(1 for _, l in veri if l == 0)}")

class SentimentDataset(Dataset):
    def __init__(self, data, tokenizer, max_len=64):
        self.data = data
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        text, label = self.data[idx]
        encoding = self.tokenizer(
            text,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'label': torch.tensor(label, dtype=torch.long)
        }

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
dataset = SentimentDataset(veri, tokenizer)
loader = DataLoader(dataset, batch_size=4, shuffle=True)

print("\nDataset hazır!")
batch = next(iter(loader))
print("input_ids shape:", batch['input_ids'].shape)
print("attention_mask shape:", batch['attention_mask'].shape)
print("labels:", batch['label'])

Veri seti: 16 örnek
Pozitif: 8
Negatif: 8

Dataset hazır!
input_ids shape: torch.Size([4, 64])
attention_mask shape: torch.Size([4, 64])
labels: tensor([1, 1, 1, 0])


In [9]:
import torch.nn as nn
from transformers import AutoModelForSequenceClassification
from torch.optim import AdamW

# DistilBERT'i yükle — 2 sınıf için
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

optimizer = AdamW(model.parameters(), lr=2e-5)
criterion = nn.CrossEntropyLoss()

print("Model yüklendi, eğitim başlıyor...")
print(f"Toplam parametre: {sum(p.numel() for p in model.parameters()):,}")

loss_history = []

for epoch in range(10):
    model.train()
    total_loss, correct = 0, 0

    for batch in loader:
        optimizer.zero_grad()

        output = model(
            input_ids=batch['input_ids'],
            attention_mask=batch['attention_mask']
        )

        loss = criterion(output.logits, batch['label'])
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        correct += (output.logits.argmax(1) == batch['label']).sum().item()

    avg_loss = total_loss / len(loader)
    acc = correct / len(dataset)
    loss_history.append(avg_loss)

    print(f"Epoch {epoch+1:2d}/10 | Loss: {avg_loss:.4f} | Acc: {acc:.2%}")

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 2455.29it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model yüklendi, eğitim başlıyor...
Toplam parametre: 66,955,010
Epoch  1/10 | Loss: 0.7167 | Acc: 50.00%
Epoch  2/10 | Loss: 0.6926 | Acc: 43.75%
Epoch  3/10 | Loss: 0.6484 | Acc: 56.25%
Epoch  4/10 | Loss: 0.5596 | Acc: 93.75%
Epoch  5/10 | Loss: 0.4540 | Acc: 100.00%
Epoch  6/10 | Loss: 0.3515 | Acc: 100.00%
Epoch  7/10 | Loss: 0.2343 | Acc: 100.00%
Epoch  8/10 | Loss: 0.1604 | Acc: 100.00%
Epoch  9/10 | Loss: 0.1063 | Acc: 100.00%
Epoch 10/10 | Loss: 0.0814 | Acc: 100.00%


In [10]:
model.eval()

yeni_cümleler = [
    "I'm so happy with this, truly excellent!",
    "Worst product ever, complete disappointment.",
    "It's okay, nothing special but works fine.",
    "Absolutely stunning, exceeded all expectations!",
    "Broke after one use, total garbage."
]

print("Fine-tuned BERT Tahminleri:")
print("=" * 55)

with torch.no_grad():
    for cümle in yeni_cümleler:
        encoding = tokenizer(
            cümle,
            max_length=64,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        output = model(
            input_ids=encoding['input_ids'],
            attention_mask=encoding['attention_mask']
        )
        probs = torch.softmax(output.logits, dim=1)
        pred = output.logits.argmax(1).item()
        güven = probs[0][pred].item()

        emoji = "⭐" if pred == 1 else "💔"
        label = "POZİTİF" if pred == 1 else "NEGATİF"
        print(f"\n{emoji} {label} ({güven:.2%})")
        print(f"   '{cümle}'")

Fine-tuned BERT Tahminleri:

⭐ POZİTİF (94.05%)
   'I'm so happy with this, truly excellent!'

💔 NEGATİF (92.34%)
   'Worst product ever, complete disappointment.'

⭐ POZİTİF (60.94%)
   'It's okay, nothing special but works fine.'

⭐ POZİTİF (88.67%)
   'Absolutely stunning, exceeded all expectations!'

💔 NEGATİF (94.91%)
   'Broke after one use, total garbage.'
